In [ ]:
# This code was adapted from https://github.com/MarvinIRW/Assessing-Answer-Accuracy-Hallucination-and-Document-Relevance-in-virtUOS-Chatbot/tree/main/code/eval

import os
import pandas as pd
import json
from sacrebleu.metrics import BLEU

def compute_sentence_bleu(
    df: pd.DataFrame,
    reference_col: str,
    hypothesis_col: str,
    question_id_col: str,
    output_csv_path: str,
    mean_csv_path=None,
    dataset_lang=None,
    bleu_metric=None
) -> pd.DataFrame:
    """
    Compute sentence-level BLEU for each row in the given DataFrame.
    
    Args:
        df (pd.DataFrame): The data containing references & hypotheses.
        reference_col (str): Name of the column with reference/human texts.
        hypothesis_col (str): Name of the column with system/hypothesis texts.
        question_id_col (str): Name of the column with question IDs (for alignment).
        output_csv_path (str): Where to save the resulting DataFrame.
        mean_csv_path (str): Where to find the mean evaluation scores.
        dataset_lang (str): The language of the dataset (de or en).
        bleu_metric (sacrebleu.metrics.BLEU, optional): 
            BLEU object to use. If None, create a new one with effective_order=True.

    Returns:
        pd.DataFrame: A new DataFrame containing sentence-level BLEU results.
    """
    if bleu_metric is None:
        bleu_metric = BLEU(effective_order=True)
    
    references = df[reference_col].astype(str).tolist()
    hypotheses = df[hypothesis_col].astype(str).tolist()
    
    bleu_scores = []
    bleu_1gram_precision = []
    bleu_2gram_precision = []
    bleu_3gram_precision = []
    bleu_4gram_precision = []
    bleu_bp = []
    bleu_sys_len = []
    bleu_ref_len = []

    for hyp, ref in zip(hypotheses, references):
        result = bleu_metric.sentence_score(hyp, [ref])
        bleu_scores.append(result.score)  # BLEU on a 0–100 scale
        bleu_1gram_precision.append(result.precisions[0])
        bleu_2gram_precision.append(result.precisions[1])
        bleu_3gram_precision.append(result.precisions[2])
        bleu_4gram_precision.append(result.precisions[3])
        bleu_bp.append(result.bp)
        bleu_sys_len.append(result.sys_len)
        bleu_ref_len.append(result.ref_len)

    # Build a new DataFrame of BLEU results
    bleu_df = pd.DataFrame()
    bleu_df[question_id_col] = df[question_id_col].values
    bleu_df['BLEU'] = bleu_scores
    bleu_df['BLEU_1gram_prec'] = bleu_1gram_precision
    bleu_df['BLEU_2gram_prec'] = bleu_2gram_precision
    bleu_df['BLEU_3gram_prec'] = bleu_3gram_precision
    bleu_df['BLEU_4gram_prec'] = bleu_4gram_precision
    bleu_df['BLEU_BP'] = bleu_bp
    bleu_df['BLEU_sys_len'] = bleu_sys_len
    bleu_df['BLEU_ref_len'] = bleu_ref_len
    
    # Compute the average (macro) of sentence-level BLEU
    avg_bleu = bleu_df['BLEU'].mean()
    print(f"Average sentence-level BLEU for {output_csv_path}: {avg_bleu:.2f}")
    print(f"BLEU signature: {bleu_metric.get_signature()}")

    # Save to CSV
    bleu_df.to_csv(output_csv_path, index=False)
    print(f"Saved BLEU results to: {output_csv_path}\n")

    if mean_csv_path is not None and os.path.exists(mean_csv_path) and dataset_lang is not None:
        # save the mean evaluation scores
        mean_eval = pd.read_csv(mean_csv_path)
        # add row to the mean_eval df
        if f"BLEU_{dataset_lang}" not in mean_eval["metric"].values:
            mean_eval = pd.concat([mean_eval, pd.DataFrame([{"metric": f"BLEU_{dataset_lang}", "value": avg_bleu}])], ignore_index=True)
        mean_eval.to_csv(mean_csv_path, index=False)
    
    return bleu_df


cwd = os.getcwd() # /app/eval 
# Get config
notebook_config_path = os.path.join(cwd, "notebook_config.json")
with open(notebook_config_path, "r") as f:
    config = json.load(f)

# 1. Load CSVs


# path to the csv containing the data to evaluate (Data should be in German)
csv_path_de = config.get("csv_path_de", os.path.join(cwd, 'results/llm_judge_results_de.csv'))
#csv_path_en = os.path.join(cwd, '../../data/short_dataset_en.csv')
csv_path_mean = config.get("csv_path_mean"+ "/mean_eval_blue_de.csv", os.path.join(cwd, 'results/mean_eval_blue_de.csv'))
df_original_de = pd.read_csv(csv_path_de)
#df_original_en = pd.read_csv(csv_path_en)

# 3. Evaluate German
output_csv_de = config.get("output_csv_de"+ "/blue_evaluation_de.cs", os.path.join(cwd, 'results/blue_evaluation_de.csv'))
bleu_df_de = compute_sentence_bleu(
    df=df_original_de,
    reference_col='human_answer',
    hypothesis_col='chatbot_answer',
    question_id_col='question_id_q',
    output_csv_path=output_csv_de,
    mean_csv_path=csv_path_mean,
    dataset_lang='de'
)
# 4. Evaluate English
# output_csv_en = os.path.join(cwd, '../../data/eval/bleu_evaluation_en.csv')
# bleu_df_en = compute_sentence_bleu(
#     df=df_original_en,
#     reference_col='human_answer_en',
#     hypothesis_col='chatbot_answer_en',
#     question_id_col='question_id_q',
#     output_csv_path=output_csv_en,
#     mean_csv_path=csv_path_mean,
#     dataset_lang='en'
# )